# Parte 2: Classificador de Risco Cardíaco

**Projeto CardioIA / Fase 2**

Luiz Felipe Alves Gomes | 2TIAOA 2026/1

## O que esse notebook faz

Treina um modelo de Machine Learning pra classificar frases de pacientes em "alto risco" ou "baixo risco" cardíaco. Simula o funcionamento de sistemas de triagem automatizada usados em hospitais pra priorizar atendimentos.

O fluxo é:
1. Carregar o dataset de frases rotuladas
2. Transformar texto em vetores numéricos com TF-IDF
3. Treinar dois modelos (Decision Tree e Logistic Regression)
4. Comparar a acurácia e testar com frases novas

## 1. Carregando o dataset

In [ ]:
!pip install pandas scikit-learn numpy -q

In [ ]:
import pandas as pd

df = pd.read_csv('../data/dataset_risco.csv')

print(f'Total de frases: {len(df)}')
print(f'\nDistribuição:')
print(df['situacao'].value_counts())
print(f'\nExemplos:')
for _, row in df.head(3).iterrows():
    print(f'  [{row["situacao"]}] {row["frase"]}')

## 2. TF-IDF: transformando texto em números

Modelos de ML não entendem texto, só números. O TF-IDF (Term Frequency - Inverse Document Frequency) resolve isso transformando cada frase num vetor numérico.

A lógica: palavras que aparecem muito numa frase mas pouco no resto do dataset ganham peso alto. Então "dor no peito" em frases de alto risco vai ter peso diferente de "leve cansaço" em frases de baixo risco.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Separando features (X) e labels (y)
X_texto = df['frase']
y = df['situacao']

# Aplicando TF-IDF
vectorizer = TfidfVectorizer()
X = vectorizer.fit_transform(X_texto)

print(f'Formato da matriz TF-IDF: {X.shape}')
print(f'Cada frase virou um vetor de {X.shape[1]} dimensões')
print(f'\nPalavras do vocabulário (primeiras 20):')
print(vectorizer.get_feature_names_out()[:20])

## 3. Dividindo treino e teste

Separamos 70% dos dados pra treino e 30% pra teste. O modelo nunca vê os dados de teste durante o treinamento, assim a gente consegue medir se ele realmente aprendeu ou só decorou.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f'Treino: {X_train.shape[0]} frases')
print(f'Teste:  {X_test.shape[0]} frases')
print(f'\nDistribuição no treino:')
print(y_train.value_counts())
print(f'\nDistribuição no teste:')
print(y_test.value_counts())

## 4. Treinando os modelos

Vou treinar dois classificadores pra comparar:

**Decision Tree** aprende regras do tipo "se contém X e Y, então alto risco". Fácil de interpretar mas pode decorar os dados de treino.

**Logistic Regression** calcula a probabilidade de ser alto ou baixo risco com base nos pesos de cada palavra. Geralmente generaliza melhor com poucos dados.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)
dt.fit(X_train, y_train)
dt_pred = dt.predict(X_test)

# Logistic Regression
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train, y_train)
lr_pred = lr.predict(X_test)

print('DECISION TREE')
print(f'Acurácia: {accuracy_score(y_test, dt_pred):.2%}')
print(classification_report(y_test, dt_pred))

print('\nLOGISTIC REGRESSION')
print(f'Acurácia: {accuracy_score(y_test, lr_pred):.2%}')
print(classification_report(y_test, lr_pred))

## 5. Testando com frases novas

Agora o teste real: frases que o modelo nunca viu.

In [ ]:
frases_novas = [
    "sinto uma dor forte no peito e estou suando frio",
    "tive um leve cansaço depois da caminhada",
    "meu coração está disparado e sinto falta de ar intensa",
    "senti uma pontada rápida no peito mas já passou",
    "desmaiei e quando acordei estava com dor no peito",
    "estou com dor de cabeça depois de ficar no sol",
]

# Vetorizando com o mesmo TF-IDF do treino
X_novas = vectorizer.transform(frases_novas)

# Usando Logistic Regression (geralmente melhor com poucos dados)
predicoes = lr.predict(X_novas)
probabilidades = lr.predict_proba(X_novas)

print('Teste com frases novas (Logistic Regression):')
print('=' * 70)
for frase, pred, prob in zip(frases_novas, predicoes, probabilidades):
    confianca = max(prob) * 100
    print(f'\n"{frase}"')
    print(f'  Classificação: {pred} ({confianca:.0f}% de confiança)')

## 6. Palavras mais importantes pra cada classe

Quais palavras o modelo aprendeu que indicam alto ou baixo risco?

In [ ]:
import numpy as np

# Pegando os coeficientes da Logistic Regression
nomes = vectorizer.get_feature_names_out()
coefs = lr.coef_[0]

# Top 10 palavras que indicam alto risco (coeficientes mais positivos)
top_alto = np.argsort(coefs)[-10:][::-1]
print('Palavras que mais indicam ALTO RISCO:')
for idx in top_alto:
    print(f'  {nomes[idx]:<20} peso: {coefs[idx]:.3f}')

# Top 10 palavras que indicam baixo risco (coeficientes mais negativos)
top_baixo = np.argsort(coefs)[:10]
print(f'\nPalavras que mais indicam BAIXO RISCO:')
for idx in top_baixo:
    print(f'  {nomes[idx]:<20} peso: {coefs[idx]:.3f}')

## 7. O que aprendi

O TF-IDF conseguiu capturar bem a diferença entre frases de alto e baixo risco. Palavras como "dor", "peito", "falta de ar", "desmaiei" ficaram com peso alto pra alto risco, enquanto "leve", "passou", "rápida" ficaram associadas a baixo risco. Faz sentido clinicamente.

Com um dataset pequeno (30 frases), a Logistic Regression tende a generalizar melhor que a Decision Tree, que pode decorar os dados de treino. Num cenário real com milhares de prontuários, a diferença seria menor.

O ponto mais interessante é que o modelo aprende sozinho quais palavras importam, sem a gente definir regras manualmente como na Parte 1. Isso é a diferença entre NLP baseado em regras e NLP com Machine Learning.